<a href="https://colab.research.google.com/github/nikolas-joyce/alpha-td-exhaustion/blob/main/notebooks/alpha_td_exhaustion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alpha: TD Sequential Exhaustion (L5/S5)

> DeMark's original exhaustion signal: 9 consecutive closes each lower than the close four bars prior (bull setup) or higher (bear setup), filtered by the TDST level. Completion signals that a move has gone too far and a reversal is probable. This notebook focuses on the standalone exhaustion signal, its sensitivity to setup count, and how it compares to the broader TD Sequential suite.

## Notebook structure
| Section | Description |
|---------|-------------|
| 0 | Install & imports |
| 1 | Config & data |
| 2 | Helper functions |
| 3 | TD Sequential engine |
| 4 | L5/S5 signal generation |
| 5 | Returns & performance |
| 6 | Per-ticker breakdown |
| 7 | Parameter sensitivity (setup count, TDST window) |
| 8 | Export signals for td-swarm |

**Run all cells top-to-bottom in a fresh Colab runtime.**


In [ ]:
# Cell 0: Installs & Imports
# ============================================================
!pip install yfinance --quiet

import time
import json
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

try:
    import yfinance as yf
except ImportError as e:
    raise ImportError('yfinance missing — restart runtime after install') from e

print('Imports complete')

## Section 1 — Config & Data

In [ ]:
# Cell 1: Config & Data
# ============================================================

TICKERS    = ['SPY','QQQ','IWM','TLT','HYG',
              'GLD','USO','UUP','EEM','VNQ','SLV','XLF','XLE']
START_DATE = '2015-01-01'
COST_BPS   = 5
SETUP_LEN   = 9
CD_LEN      = 13
HOLD_WINDOW = 63
TDST_WINDOW = 9
WF_SPLIT_DATE = '2022-01-01'

print('Fetching OHLCV...')
raw = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)

assert isinstance(raw.columns, pd.MultiIndex), \
    'Expected MultiIndex columns from yf.download'
assert set(['Close','High','Low']).issubset(
    raw.columns.get_level_values(0)), 'Missing OHLC columns'

close   = raw['Close'].dropna(how='all').ffill(limit=3)
high    = raw['High'].dropna(how='all').ffill(limit=3)
low     = raw['Low'].dropna(how='all').ffill(limit=3)
returns = close.pct_change()
idx     = close.index
high    = high.reindex(idx).ffill(limit=3)
low     = low.reindex(idx).ffill(limit=3)

keep = close.isnull().mean() < 0.20
if not keep.all():
    dropped = close.columns[~keep].tolist()
    print(f'  Dropping tickers with >20% NaN: {dropped}')
    close, high, low, returns = (
        df[keep[keep].index] for df in [close, high, low, returns])
    TICKERS = close.columns.tolist()

print(f'  Shape  : {close.shape}')
print(f'  Range  : {close.index[0].date()} -> {close.index[-1].date()}')
print(f'  Tickers: {TICKERS}')
print('Data ready')

## Section 2 — Helper Functions

In [ ]:
# Cell 2: Helper Functions
# ============================================================
# FIX: compute_atr — removed deprecated groupby(level=1, axis=1).
# FIX: apply_costs — generalised to Series or DataFrame, returns SIGNED P&L.

def compute_atr(high, low, close, period=14):
    """Average True Range — vectorised."""
    prev = close.shift(1)
    tr   = pd.DataFrame(
        np.maximum(
            np.maximum(high.values - low.values,
                       np.abs(high.values - prev.values)),
            np.abs(low.values - prev.values)),
        index=close.index, columns=close.columns)
    return tr.rolling(period, min_periods=period).mean()


def apply_costs(returns, position, cost_bps=5):
    """
    Signed net P&L after transaction costs.
    result = returns * position - |d(position)| * cost
    The sign of position is preserved — this is a SIGNED P&L series.
    Accepts Series or DataFrame; aligns index automatically.
    """
    cost     = cost_bps / 10_000
    position = position.reindex(returns.index).fillna(0)
    turnover = position.diff().abs()
    return returns * position - turnover * cost


def performance_summary(r, name=''):
    """Annualised stats for a daily return Series."""
    r = r.dropna()
    if len(r) < 21:
        return pd.Series(
            dict(zip(['Ann. Return','Ann. Vol','Sharpe','Max DD','Calmar','Hit Rate'],
                     [np.nan]*6)), name=name)
    ann     = 252
    cum     = (1 + r).cumprod()
    ann_ret = (1 + r.mean()) ** ann - 1
    ann_vol = r.std() * np.sqrt(ann)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-9 else np.nan
    mdd     = (cum / cum.cummax() - 1).min()
    calmar  = ann_ret / abs(mdd) if abs(mdd) > 1e-9 else np.nan
    hits    = (r > 0).mean()
    return pd.Series({
        'Ann. Return': ann_ret, 'Ann. Vol': ann_vol,
        'Sharpe': sharpe, 'Max DD': mdd,
        'Calmar': calmar, 'Hit Rate': hits,
    }, name=name)


def score_variant(stats):
    """Composite swarm score: Sharpe×0.4 + Calmar×0.3 + HitRate×0.3."""
    s = stats.get('Sharpe', np.nan)
    c = stats.get('Calmar', np.nan)
    h = stats.get('Hit Rate', np.nan)
    if any(np.isnan(v) for v in [s, c, h]):
        return -999.0
    return s * 0.40 + c * 0.30 + h * 0.30


print('Helpers loaded')

## Section 3 — TD Sequential Engine

In [ ]:
# Cell 3: TD Sequential Core Engine
# ============================================================
# FIX: compute_td_countdown uses numpy array indexing (3-5x faster).
# FIX: compute_tdst tdst_window argument now used throughout.
# All window parameters are arguments so the swarm can vary them.

def rolling_count(cond):
    """Consecutive True count that resets to 0 on False. Vectorised."""
    c   = cond.astype(int)
    raw = c.copy()
    for col in c.columns:
        s        = c[col]
        gid      = (~cond[col]).astype(int).cumsum()
        raw[col] = s.groupby(gid).cumsum() * s
    return raw


def compute_td_setup(close, high, low, setup_len=9):
    """TD Setup: setup_len consecutive bars of close vs close[-4]."""
    close4     = close.shift(4)
    bull_count = rolling_count(close < close4)
    bear_count = rolling_count(close > close4)
    bull_n     = (bull_count == setup_len).astype(int)
    bear_n     = (bear_count == setup_len).astype(int)
    # Perfect setup: bars n-3 and n-2 relative to bar n
    bull_perfect = ((bull_n == 1) &
                    ((low <= low.shift(3)) | (low <= low.shift(2)))).astype(int)
    bear_perfect = ((bear_n == 1) &
                    ((high >= high.shift(3)) | (high >= high.shift(2)))).astype(int)
    return {
        'bull_count': bull_count, 'bear_count': bear_count,
        'bull_setup9': bull_n,   'bear_setup9': bear_n,
        'bull_perfect': bull_perfect, 'bear_perfect': bear_perfect,
        'setup_len': setup_len,
    }


def compute_td_countdown(close, high, low,
                          bull_setup9, bear_setup9, cd_len=13):
    """TD Countdown with recycling and cancellation. Numpy array loop."""
    low2   = low.shift(2).values
    high2  = high.shift(2).values
    c_arr  = close.values
    bs_arr = bull_setup9.values
    rs_arr = bear_setup9.values
    n_bars, n_tickers = c_arr.shape
    bcc  = np.zeros((n_bars, n_tickers), dtype=np.int32)
    rcc  = np.zeros((n_bars, n_tickers), dtype=np.int32)
    bact = np.zeros((n_bars, n_tickers), dtype=np.int32)
    ract = np.zeros((n_bars, n_tickers), dtype=np.int32)
    brec = np.zeros(n_tickers, dtype=np.int32)
    rrec = np.zeros(n_tickers, dtype=np.int32)
    for j in range(n_tickers):
        b_in, b_cnt = False, 0
        for i in range(n_bars):
            if bs_arr[i,j] == 1:
                if b_in: brec[j] += 1
                b_cnt, b_in = 0, True
            if rs_arr[i,j] == 1: b_in, b_cnt = False, 0
            if b_in and b_cnt < cd_len:
                l2 = low2[i,j]
                if not np.isnan(l2) and c_arr[i,j] <= l2: b_cnt += 1
            bcc[i,j]  = b_cnt if b_in else 0
            bact[i,j] = 1     if b_in else 0
        r_in, r_cnt = False, 0
        for i in range(n_bars):
            if rs_arr[i,j] == 1:
                if r_in: rrec[j] += 1
                r_cnt, r_in = 0, True
            if bs_arr[i,j] == 1: r_in, r_cnt = False, 0
            if r_in and r_cnt < cd_len:
                h2 = high2[i,j]
                if not np.isnan(h2) and c_arr[i,j] >= h2: r_cnt += 1
            rcc[i,j]  = r_cnt if r_in else 0
            ract[i,j] = 1     if r_in else 0
    def w(a): return pd.DataFrame(a, index=close.index, columns=close.columns)
    bc, rc = w(bcc), w(rcc)
    return {
        'bull_cd_count': bc,  'bear_cd_count': rc,
        'bull_cd13': (bc == cd_len).astype(int),
        'bear_cd13': (rc == cd_len).astype(int),
        'bull_cd_active': w(bact), 'bear_cd_active': w(ract),
        'bull_recycled': dict(zip(close.columns, brec.tolist())),
        'bear_recycled': dict(zip(close.columns, rrec.tolist())),
        'cd_len': cd_len,
    }


def compute_tdst(close, high, low, bull_setup9, bear_setup9, tdst_window=9):
    """TDST Support/Resistance. FIX: now uses tdst_window argument throughout."""
    lowest_n  = low.rolling(tdst_window, min_periods=tdst_window).min()
    highest_n = high.rolling(tdst_window, min_periods=tdst_window).max()
    sup = pd.DataFrame(np.nan, index=close.index, columns=close.columns)
    res = pd.DataFrame(np.nan, index=close.index, columns=close.columns)
    sup[bull_setup9 == 1]    = lowest_n[bull_setup9 == 1]
    res[bear_setup9 == 1]    = highest_n[bear_setup9 == 1]
    sup = sup.ffill(); res = res.ffill()
    return {
        'tdst_support':    sup, 'tdst_resistance': res,
        'tdst_long_valid':  (close > sup).astype(int),
        'tdst_short_valid': (close < res).astype(int),
    }


def run_td_engine(close, high, low, setup_len=9, cd_len=13, tdst_window=9):
    """Single entry point: Setup -> Countdown -> TDST."""
    s  = compute_td_setup(close, high, low, setup_len)
    cd = compute_td_countdown(close, high, low,
                              s['bull_setup9'], s['bear_setup9'], cd_len)
    ts = compute_tdst(close, high, low,
                      s['bull_setup9'], s['bear_setup9'], tdst_window)
    return s, cd, ts


print('Running TD engine...')
t0 = time.time()
td_setup, td_countdown, td_tdst = run_td_engine(
    close, high, low,
    setup_len=SETUP_LEN, cd_len=CD_LEN, tdst_window=TDST_WINDOW)
print(f'  Done in {time.time()-t0:.1f}s')
print(f'  Bull setups : {td_setup["bull_setup9"].sum().sum()}')
print(f'  Bear setups : {td_setup["bear_setup9"].sum().sum()}')
print(f'  Bull CD13s  : {td_countdown["bull_cd13"].sum().sum()}')
print(f'  Bear CD13s  : {td_countdown["bear_cd13"].sum().sum()}')
print('TD engine ready')

## Section 4 — TD Wrapper

In [ ]:
# compute_td_signals — convenience wrapper used by signal cells
def compute_td_signals(close, high, low, setup_count, countdown_count,
                       setup_lookback, countdown_lookback, tdst_window):
    """Run full TD engine and return (td_setup, td_tdst, td_countdown, td_combo)."""
    td_setup    = compute_td_setup(close, high, low, setup_len=setup_count)
    td_cd       = compute_td_countdown(close, high, low,
                                       td_setup['bull_setup9'],
                                       td_setup['bear_setup9'],
                                       cd_len=countdown_count)
    td_tdst     = compute_tdst(close, high, low,
                                td_setup['bull_setup9'],
                                td_setup['bear_setup9'],
                                window=tdst_window)
    return td_setup, td_tdst, td_cd, None


## Section 5 — TD Variant Signal Functions

In [ ]:
# Cell 4: TD Variant Signal Functions
# ============================================================
# POLARITY:
#   +1 CONVENTIONAL: bull setup -> long,  bear setup -> short  (Demark original)
#   -1 CONTRARIAN:   bear setup -> long,  bull setup -> short  (momentum continuation)
#
# EXIT LOGIC (same for all variants):
#   Primary exit  : TDST gate goes to 0 (price crosses TDST level)
#   Secondary exit: rolling hold_window expires (time stop)
#   Silent re-entry caveat: if TDST recovers within hold_window bars,
#     old entry is still in rolling window -> position re-opens.
#     A real Demark trader requires a fresh setup bar.

POLARITY_LABELS = {+1: 'conventional', -1: 'contrarian'}


def _carry_pos(entry, gate, hold_window):
    """Carry entry forward for hold_window bars, gate by TDST, shift 1."""
    return (entry.rolling(hold_window, min_periods=1).max()
            * gate).clip(0, 1).shift(1).fillna(0)


def td_v1_setup9(td_setup, td_tdst, hold_window=63, polarity=+1):
    """V1: Setup bar N only."""
    if polarity == +1:
        le = ((td_setup['bull_setup9']==1)&(td_tdst['tdst_long_valid']==1)).astype(int)
        se = ((td_setup['bear_setup9']==1)&(td_tdst['tdst_short_valid']==1)).astype(int)
        lg, sg = td_tdst['tdst_long_valid'], td_tdst['tdst_short_valid']
    else:
        le = ((td_setup['bear_setup9']==1)&(td_tdst['tdst_short_valid']==1)).astype(int)
        se = ((td_setup['bull_setup9']==1)&(td_tdst['tdst_long_valid']==1)).astype(int)
        lg, sg = td_tdst['tdst_short_valid'], td_tdst['tdst_long_valid']
    return _carry_pos(le, lg, hold_window), _carry_pos(se, sg, hold_window)


def td_v2_countdown13(td_countdown, td_tdst, hold_window=126, polarity=+1):
    """V2: Countdown bar N only."""
    if polarity == +1:
        le = ((td_countdown['bull_cd13']==1)&(td_tdst['tdst_long_valid']==1)).astype(int)
        se = ((td_countdown['bear_cd13']==1)&(td_tdst['tdst_short_valid']==1)).astype(int)
        lg, sg = td_tdst['tdst_long_valid'], td_tdst['tdst_short_valid']
    else:
        le = ((td_countdown['bear_cd13']==1)&(td_tdst['tdst_short_valid']==1)).astype(int)
        se = ((td_countdown['bull_cd13']==1)&(td_tdst['tdst_long_valid']==1)).astype(int)
        lg, sg = td_tdst['tdst_short_valid'], td_tdst['tdst_long_valid']
    return _carry_pos(le, lg, hold_window), _carry_pos(se, sg, hold_window)


def td_v3_both(td_setup, td_countdown, td_tdst, hold_window=126, polarity=+1):
    """V3: Half on setup, full on countdown. FIX: short side uses bear keys."""
    def _b(sk, ck, gk):
        se = ((td_setup[sk]==1)&(td_tdst[gk]==1)).astype(int)
        ce = ((td_countdown[ck]==1)&(td_tdst[gk]==1)).astype(int)
        return ((se.rolling(hold_window,min_periods=1).max()*0.5 +
                 ce.rolling(hold_window,min_periods=1).max()*0.5)
                * td_tdst[gk]).clip(0,1).shift(1).fillna(0)
    if polarity == +1:
        return _b('bull_setup9','bull_cd13','tdst_long_valid'), \
               _b('bear_setup9','bear_cd13','tdst_short_valid')
    else:
        return _b('bear_setup9','bear_cd13','tdst_short_valid'), \
               _b('bull_setup9','bull_cd13','tdst_long_valid')


def td_v4_separate(td_setup, td_countdown, td_tdst,
                   hold_window=63, polarity=+1):
    """V4: Setup and countdown as independent alphas. Returns four frames."""
    sl, ss = td_v1_setup9(td_setup, td_tdst, hold_window, polarity)
    cl, cs = td_v2_countdown13(td_countdown, td_tdst, hold_window*2, polarity)
    return sl, ss, cl, cs


# -- Default conventional signals ----------------------------------
print('Computing variant signals (conventional polarity)...')
v1_long,  v1_short = td_v1_setup9(td_setup, td_tdst, HOLD_WINDOW, +1)
v2_long,  v2_short = td_v2_countdown13(td_countdown, td_tdst, HOLD_WINDOW*2, +1)
v3_long,  v3_short = td_v3_both(td_setup, td_countdown, td_tdst, HOLD_WINDOW*2, +1)
v4_setup_long, v4_setup_short, v4_cd_long, v4_cd_short = \
    td_v4_separate(td_setup, td_countdown, td_tdst, HOLD_WINDOW, +1)
print('Done. Density:')
for lbl, sig in [('V1 Long',v1_long),('V1 Short',v1_short),
                 ('V2 Long',v2_long),('V2 Short',v2_short),
                 ('V3 Long',v3_long),('V3 Short',v3_short)]:
    print(f'  {lbl:12s}: {sig.mean().mean():.1%}')

## Section 6 — Signal Generation

In [ ]:
# TD Exhaustion Signal (L5/S5)
# Uses TD Setup bar-9 with TDST gate — same as V1 conventional.
# L5: bull setup9 completes + TDST long valid → long
# S5: bear setup9 completes + TDST short valid → short
# This notebook explores the standalone exhaustion signal and its sensitivity.

ALPHA_LABEL = 'TD_Exhaustion'

td_setup_d, _, _, tdst_d = compute_td_signals(
    close, high, low,
    setup_count=SETUP_LEN, countdown_count=CD_LEN,
    setup_lookback=4, countdown_lookback=2,
    tdst_window=TDST_WINDOW,
)

v_long, v_short = td_v1_setup9(td_setup_d, tdst_d, HOLD_WINDOW, polarity=+1)

for label, sig in [('L5 long', v_long), ('S5 short', v_short)]:
    print(f'{label}: {int(sig.sum().sum())} signals across {len(sig.columns)} tickers')


## Section 7 — Returns & Performance

In [ ]:
# Returns & Performance — single-alpha

def compute_alpha_returns(long_sig, short_sig, returns, label, cost_bps=COST_BPS):
    def _side(sig, direction, side_label):
        pos = sig.reindex(returns.index).fillna(0) * direction
        net = apply_costs(returns, pos, cost_bps)
        n   = sig.sum(axis=1).replace(0, float('nan'))
        pnl = net.sum(axis=1)
        return (pnl / n).fillna(0).rename(f'{label}_{side_label}')
    l_ret    = _side(long_sig,  +1, 'Long')
    s_ret    = _side(short_sig, -1, 'Short')
    combined = ((l_ret + s_ret) / 2).rename(label)
    return l_ret, s_ret, combined

a_long_ret, a_short_ret, a_combined = compute_alpha_returns(
    v_long, v_short, returns, ALPHA_LABEL)

print(f'Alpha: {ALPHA_LABEL}')
print(performance_summary(a_long_ret,  name='Long side').round(3).to_string())
print()
print(performance_summary(a_short_ret, name='Short side').round(3).to_string())
print()
print(performance_summary(a_combined,  name='Combined').round(3).to_string())

# Equity curve
cum = (1 + a_combined).cumprod()
fig, ax = plt.subplots(figsize=(14, 5))
cum.plot(ax=ax, title=f'{ALPHA_LABEL} — Equity Curve')
ax.set_ylabel('Cumulative return')
plt.tight_layout()
plt.show()


## Section 8 — Per-Ticker Breakdown

In [ ]:
# Per-Ticker Performance Breakdown

def per_ticker_sharpe_single(long_sig, short_sig, returns, label):
    sharpes = {}
    for ticker in TICKERS:
        lp  = long_sig[ticker].reindex(returns.index).fillna(0)
        sp  = short_sig[ticker].reindex(returns.index).fillna(0)
        ln  = apply_costs(returns[[ticker]], lp.to_frame(), COST_BPS)[ticker]
        sn  = apply_costs(returns[[ticker]], (sp*-1).to_frame(), COST_BPS)[ticker]
        cr  = ((ln + sn)/2).dropna()
        std = cr.std()
        sharpes[ticker] = (cr.mean()/std*np.sqrt(252) if std>1e-9 else float('nan'))
    return pd.Series(sharpes, name=label)

ts = per_ticker_sharpe_single(v_long, v_short, returns, ALPHA_LABEL)
print(f'Per-Ticker Sharpe ({ALPHA_LABEL}):')
print(ts.sort_values(ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
ts.sort_values().plot(kind='barh', ax=ax, title=f'{ALPHA_LABEL} — Per-Ticker Sharpe')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()


## Section 9 — Parameter Sensitivity

In [ ]:
# Parameter Sensitivity — TD Exhaustion
# Vary setup count (bar N) and TDST window

import itertools
sweep_rows = []
for sc, tw in itertools.product([7, 8, 9, 10, 11], [7, 9, 13]):
    try:
        td_s, _, _, tdst_s = compute_td_signals(
            close, high, low, setup_count=sc, countdown_count=CD_LEN,
            setup_lookback=4, countdown_lookback=2, tdst_window=tw)
        vl, vs = td_v1_setup9(td_s, tdst_s, HOLD_WINDOW, polarity=+1)
        _, _, comb = compute_alpha_returns(vl, vs, returns, f'sc{sc}_tw{tw}')
        sweep_rows.append({**performance_summary(comb).to_dict(),
                           'setup_count': sc, 'tdst_window': tw})
    except Exception as e:
        print(f'sc={sc} tw={tw}: {e}')
sweep_df = pd.DataFrame(sweep_rows).sort_values('Sharpe', ascending=False)
print(sweep_df[['setup_count','tdst_window','Sharpe','Ann. Return','Max DD']].round(3).to_string())


## Section 10 — Export Signals for Swarm

In [ ]:
import os, pickle
os.makedirs('exports', exist_ok=True)
with open('exports/alpha_td_exhaustion.pkl', 'wb') as f:
    pickle.dump({'long': v_long, 'short': v_short}, f)
print(f'Signals exported -> exports/alpha_td_exhaustion.pkl')
